# NB 00 — Environment Setup & Verification

**Phase 1 — Tasks 1.1 / 1.2**

This notebook:
- Mounts Google Drive
- Installs all required packages
- Sets CAMELTOOLS_DATA (centralised in src/utils.py — no manual env var needed)
- Initialises a SparkSession
- Verifies the environment


## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


Mounted at /content/drive
Drive mounted.


## 1. Install dependencies

In [ ]:
%%capture
!pip install pyspark==3.5.1 datasets==2.19.0 camel-tools pyarabic
!pip install nltk scipy wordcloud plotly kafka-python
!pip install torch transformers sentencepiece sentence-transformers
!pip install arabic-reshaper python-bidi
!apt-get -qq update && apt-get -qq install -y fonts-noto-core fonts-hosny-amiri


## 2. Download NLTK data

In [ ]:
import nltk
for pkg in ['punkt', 'stopwords']:
    nltk.download(pkg, quiet=True)
print('NLTK data ready.')


NLTK data ready.


## 3. Add project src to Python path

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('sys.path updated.')
print('PROJECT_ROOT:', PROJECT_ROOT)


sys.path updated.
PROJECT_ROOT: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection


## 4. Create SparkSession

CAMELTOOLS_DATA is set automatically inside `src/utils.py` using the
centralised `CAMEL_CACHE` constant — no manual `os.environ` call needed here.

In [ ]:
from src.utils import create_spark_session, PROJECT_ROOT, CAMEL_CACHE

spark = create_spark_session(
    app_name='ArabicAIDetection_Setup',
    executor_memory='4g',
    driver_memory='4g',
)
print(f'Spark {spark.version} ready')
print(f'CAMEL_CACHE: {CAMEL_CACHE}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


Spark 3.5.1 ready
CAMEL_CACHE: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/.camel_cache
PROJECT_ROOT: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection


## 5. Download CAMeL Tools data (first run only)

In [ ]:
# This downloads ~1 GB of morphological data to CAMEL_CACHE on GDrive.
# Subsequent sessions load from cache instantly.
!camel_data -i all


The following packages will be installed: 'dialectid-model26', 'ner-arabert', 'disambig-mle-calima-msa-r13', 'morphology-db-glf-01', 'disambig-mle-calima-egy-r13', 'disambig-ranking-cache-calima-lev-01', 'disambig-bert-unfactored-glf', 'morphology-db-msa-s31', 'disambig-ranking-cache-calima-glf-01', 'disambig-bert-unfactored-lev', 'dialectid-model6', 'morphology-db-egy-r13', 'disambig-bert-unfactored-egy', 'disambig-ranking-cache-calima-msa-r13', 'morphology-db-msa-r13', 'disambig-bert-unfactored-msa', 'disambig-ranking-cache-calima-egy-r13', 'morphology-db-lev-01', 'sentiment-analysis-mbert', 'sentiment-analysis-arabert'
Extracting package 'dialectid-model26': 100% 371M/371M [00:11<00:00, 32.4MB/s]
Extracting package 'ner-arabert': 100% 542M/542M [00:08<00:00, 63.9MB/s]
Extracting package 'disambig-mle-calima-msa-r13': 100% 88.7M/88.7M [00:01<00:00, 84.9MB/s]
Extracting package 'morphology-db-glf-01': 100% 7.98M/7.98M [00:00<00:00, 114MB/s]
Extracting package 'disambig-mle-calima-egy-

## 6. Verify environment

In [ ]:
import platform, pyspark, torch, transformers

checks = {
    'Python':       platform.python_version(),
    'PySpark':      pyspark.__version__,
    'PyTorch':      torch.__version__,
    'Transformers': transformers.__version__,
    'GDrive root':  str(PROJECT_ROOT),
}
for k, v in checks.items():
    print(f'  {k:15s}: {v}')
print('\n Environment ready.')


  Python         : 3.12.13
  PySpark        : 3.5.1
  PyTorch        : 2.10.0+cpu
  Transformers   : 4.43.4
  GDrive root    : /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection

✅ Environment ready.


## 7. Spark sanity check

In [ ]:
test_df = spark.range(10).withColumnRenamed('id', 'row_id')
test_df.show()
print(f'Partitions: {test_df.rdd.getNumPartitions()}')


+------+
|row_id|
+------+
|     0|
|     1|
|     2|
|     3|
|     4|
|     5|
|     6|
|     7|
|     8|
|     9|
+------+

Partitions: 2


## 8. Notes for Colab

- **Session limit**: Free Colab disconnects after ~90 min idle or ~12 hr total.
- **After reconnecting**: Re-run cells 0–4 (mount → install → nltk → path → spark).
- **Checkpoints**: Every downstream notebook saves results to `data/processed/` as
  Parquet. Use `checkpoint_exists(name)` before re-running expensive cells.
- **CAMeL data**: Cached to GDrive — only downloads once.
- **FORCE_REBUILD flags**: Default to `False` in all notebooks. Set `True` only
  after changing `data_preparation.py` or `feature_engineering.py`.